# PEFT Fine-Tuning

In this notebook, we fine-tune the Voxtral multimodal model using
Parameter-Efficient Fine-Tuning (PEFT) with LoRA adapters.

We use:
- EmoBox-preprocessed IEMOCAP training data
- Audio-only and audio+text supervision
- Hugging Face Transformers + PEFT
- Google Colab GPU

The goal is to improve emotion classification performance
over zero-shot inference while keeping training lightweight and reproducible.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Setup & Reproducibility

This cell imports core libraries, sets random seeds for reproducibility,
and defines key project paths used throughout the fine-tuning notebook.

In [2]:
import os
import sys
import json
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# Paths
ADSP_ROOT = "/content/drive/MyDrive/adsp"
EMOBOX_ROOT = os.path.join(ADSP_ROOT, "EmoBox")

DATA_DIR = ADSP_ROOT                        # where "downloads/..." lives
META_DIR = os.path.join(EMOBOX_ROOT, "data")# parent dir containing esd/

print("ADSP_ROOT :", ADSP_ROOT)
print("EMOBOX_ROOT:", EMOBOX_ROOT)
print("DATA_DIR  :", DATA_DIR)
print("META_DIR  :", META_DIR)

assert os.path.exists(ADSP_ROOT), "ADSP_ROOT not found"
assert os.path.exists(EMOBOX_ROOT), "EMOBOX_ROOT not found"
assert os.path.exists(os.path.join(META_DIR, "esd")), "ESD metadata folder not found"

Device: cpu
ADSP_ROOT : /content/drive/MyDrive/adsp
EMOBOX_ROOT: /content/drive/MyDrive/adsp/EmoBox
DATA_DIR  : /content/drive/MyDrive/adsp
META_DIR  : /content/drive/MyDrive/adsp/EmoBox/data


## Load ESD with EmoBox

This cell loads the ESD train and test splits using EmoBox's EmoDataset.
We keep the split definitions consistent with the project metadata.

In [3]:
# ============================================================
# Load ESD dataset correctly with EmoBox
# ============================================================

import os
import sys

# ------------------------------------------------------------
# Set working directory
# ------------------------------------------------------------
os.chdir("/content/drive/MyDrive/adsp")
print("Current working directory:", os.getcwd())

# ------------------------------------------------------------
# Add EmoBox to Python path
# ------------------------------------------------------------
EMOBOX_ROOT = "/content/drive/MyDrive/adsp/EmoBox"
sys.path.append(EMOBOX_ROOT)

# ------------------------------------------------------------
# Define dataset paths
# ------------------------------------------------------------
DATA_DIR = "/content/drive/MyDrive/adsp"              # contains downloads/
META_DIR = "/content/drive/MyDrive/adsp/EmoBox/data"  # contains esd/, iemocap/

# Sanity checks
assert os.path.exists(DATA_DIR), "DATA_DIR does not exist"
assert os.path.exists(META_DIR), "META_DIR does not exist"
assert os.path.exists(os.path.join(DATA_DIR, "downloads", "esd")), "ESD audio not found"
assert os.path.exists(os.path.join(META_DIR, "esd")), "ESD metadata not found"

print("Paths verified.")

# ------------------------------------------------------------
# Load ESD using EmoBox
# ------------------------------------------------------------
from EmoBox.EmoDataset import EmoDataset

esd_train = EmoDataset(
    dataset="esd",
    data_dir=DATA_DIR,
    meta_data_dir=META_DIR,
    split="train"
)

esd_test = EmoDataset(
    dataset="esd",
    data_dir=DATA_DIR,
    meta_data_dir=META_DIR,
    split="test"
)

# ------------------------------------------------------------
# Final confirmation
# ------------------------------------------------------------
print("ESD train samples:", len(esd_train))
print("ESD test samples :", len(esd_test))

Current working directory: /content/drive/.shortcut-targets-by-id/1gH4RO3pZc-gbQXU4xk8xp-4rwW5XozbA/adsp
Paths verified.
since there is no official valid data, use random split for train valid split, with a ratio of [80, 20]
load in 28000 samples, only 28000 exists in data dir /content/drive/MyDrive/adsp/EmoBox/data
load in 7000 samples, only 7000 exists in data dir /content/drive/MyDrive/adsp/EmoBox/data
Num. training samples 28000
Num. valid samples 0
Num. test samples 7000
Using label_map {'Neutral': 'Neutral', 'Angry': 'Angry', 'Happy': 'Happy', 'Sad': 'Sad', 'Surprise': 'Surprise'}
since there is no official valid data, use random split for train valid split, with a ratio of [80, 20]
load in 28000 samples, only 28000 exists in data dir /content/drive/MyDrive/adsp/EmoBox/data
load in 7000 samples, only 7000 exists in data dir /content/drive/MyDrive/adsp/EmoBox/data
Num. training samples 28000
Num. valid samples 0
Num. test samples 7000
Using label_map {'Neutral': 'Neutral', 'Angry'

## Filter ESD to English-Only Speakers

ESD contains multiple speakers. We restrict the dataset to English speakers
(0011–0020) to ensure language consistency for fine-tuning.

In [4]:
EN_SPEAKERS = {f"{i:04d}" for i in range(11, 21)}  # 0011..0020

def filter_esd_english(dataset):
    filtered = []
    for item in dataset.data_list:
        # wav looks like: downloads/esd/0001/Neutral/0001_000001.wav
        speaker = item["wav"].split("/")[2]
        if speaker in EN_SPEAKERS:
            filtered.append(item)
    dataset.data_list = filtered
    return dataset

esd_train = filter_esd_english(esd_train)
esd_test  = filter_esd_english(esd_test)

print("ESD train (English-only):", len(esd_train))
print("ESD test  (English-only):", len(esd_test))

ESD train (English-only): 12250
ESD test  (English-only): 5250


## Define Final Label Space

We fine-tune on a 4-class emotion space:
Angry, Happy, Sad, Neutral.

We exclude Surprise to keep a consistent label set.

In [5]:
EMOTIONS = ["Angry", "Happy", "Sad", "Neutral"]

def keep_4class(dataset):
    dataset.data_list = [x for x in dataset.data_list if x["emo"] in EMOTIONS]
    return dataset

esd_train = keep_4class(esd_train)
esd_test  = keep_4class(esd_test)

print("ESD train (4-class):", len(esd_train))
print("ESD test  (4-class):", len(esd_test))

ESD train (4-class): 9800
ESD test  (4-class): 4200


## Build PEFT Training Records

We convert EmoBox samples into lightweight records for Hugging Face training.
Each record contains:
- absolute audio file path
- instruction prompt
- target label (as short text)

In [6]:
def build_prompt():
    return (
        "You are an expert at recognizing emotions from speech. "
        "Listen to the audio and output only one label: Angry, Happy, Sad, or Neutral."
    )

def build_records(dataset):
    records = []
    for item in dataset.data_list:
        records.append({
            "audio": os.path.join(DATA_DIR, item["wav"]),
            "prompt": build_prompt(),
            "label": item["emo"]
        })
    return records

train_records = build_records(esd_train)
test_records  = build_records(esd_test)

print("Train records:", len(train_records))
print("Test records :", len(test_records))

# Quick sanity check (first record)
train_records[0]

Train records: 9800
Test records : 4200


{'audio': '/content/drive/MyDrive/adsp/downloads/esd/0012/Neutral/0012_000001.wav',
 'prompt': 'You are an expert at recognizing emotions from speech. Listen to the audio and output only one label: Angry, Happy, Sad, or Neutral.',
 'label': 'Neutral'}

## Convert Records into Hugging Face Datasets

We create Dataset objects so we can efficiently map preprocessing
and feed batches into the Trainer.

In [7]:
from datasets import Dataset

train_ds = Dataset.from_list(train_records)
test_ds  = Dataset.from_list(test_records)

print(train_ds)
print(test_ds)

Dataset({
    features: ['audio', 'prompt', 'label'],
    num_rows: 9800
})
Dataset({
    features: ['audio', 'prompt', 'label'],
    num_rows: 4200
})


## Load Voxtral in 8-bit Precision (PEFT Setup)

To fit Voxtral on limited Colab hardware, we load the base model
in 8-bit precision using bitsandbytes.

The base model weights remain frozen and memory-efficient,
while LoRA adapters (added later) will be trained in full precision.
This is the standard setup for PEFT fine-tuning.

In [8]:
!pip install -q bitsandbytes
!pip install -q mistral-common

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 9.0 MB/s eta 0:00:00


In [9]:
# ============================================================
# Load Voxtral in 8-bit for PEFT
# ============================================================

import torch
from transformers import AutoProcessor, AutoModel

LOCAL_MODEL_DIR = "/content/drive/MyDrive/adsp/models/voxtral-mini-3b"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# Load processor (CPU, lightweight)
processor = AutoProcessor.from_pretrained(
    LOCAL_MODEL_DIR,
    trust_remote_code=True
)

# Load model in 8-bit (CRITICAL)
model = AutoModel.from_pretrained(
    LOCAL_MODEL_DIR,
    trust_remote_code=True,
    load_in_8bit=True,     # 🔑 key line
    device_map="auto"     # 🔑 key line
)

print("Voxtral loaded in 8-bit successfully.")

Device: cpu


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Voxtral loaded in 8-bit successfully.


### Data Collator for Voxtral (Audio-Only, EmoBox-Compatible)

At this stage, we define a **custom data collator** responsible for preparing batches during training.

Why a custom collator is required:
- EmoBox datasets store audio as **file paths** (key: `wav`), not preloaded tensors.
- Voxtral’s tokenizer is **not compatible** with Hugging Face’s default collation logic.
- Emotion labels are **strings**, not numeric class IDs.

What this collator does:
1. Loads raw audio files from disk using the `wav` paths.
2. Processes audio **only** using Voxtral’s audio processor (no transcripts).
3. Tokenizes emotion labels manually as text targets.
4. Returns properly formatted tensors for the Trainer.

This design avoids tokenizer crashes and preserves an audio-centric emotion recognition pipeline.

In [37]:
import soundfile as sf
import torch

class VoxtralDataCollator:
    def __init__(self, processor, device, max_label_tokens=4):
        self.feature_extractor = processor.feature_extractor
        self.tokenizer = processor.tokenizer
        self.device = device
        self.max_label_tokens = max_label_tokens

    def __call__(self, batch):
        audios = []
        labels = []
        srs = []

        for ex in batch:
            audio, sr = sf.read(ex["audio"])   # ✅ key is "audio"
            audios.append(audio)
            labels.append(ex["label"])
            srs.append(sr)

        # assume consistent SR within a batch
        sr0 = srs[0]

        # Audio → features
        audio_inputs = self.feature_extractor(
            audios,
            sampling_rate=sr0,
            padding=True,
            return_tensors="pt",
        )

        # Labels → tokens (targets)
        label_inputs = self.tokenizer(
            labels,
            padding=True,
            truncation=True,
            max_length=self.max_label_tokens,
            return_tensors="pt",
        )

        audio_inputs["labels"] = label_inputs.input_ids

        return {k: v.to(self.device) for k, v in audio_inputs.items()}


device = "cuda" if torch.cuda.is_available() else "cpu"
data_collator = VoxtralDataCollator(processor=processor, device=device)
print("✅ Collator ready.")

✅ Collator ready.


In [38]:
batch = [train_ds[i] for i in range(2)]
out = data_collator(batch)

for k, v in out.items():
    print(k, v.shape)

input_features torch.Size([2, 128, 286])
labels torch.Size([2, 4])


## Configure PEFT with LoRA

We add LoRA adapters to the base model so that only a small number
of parameters are trained. The base Voxtral weights remain frozen.

In [21]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    task_type="CAUSAL_LM",  # common setting for label generation
    target_modules=["q_proj", "v_proj"]  # typical transformer attention modules
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 4,014,080 || all params: 4,680,285,184 || trainable%: 0.0858


## Training Hyperparameters

We configure a Colab-friendly training setup using gradient accumulation.
This enables effective batch sizes even with limited GPU memory.


In [23]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./voxtral-esd-peft",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=3,
    fp16=(device == "cuda"),
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    report_to="none"
)

## Run PEFT Fine-Tuning

We launch training using Hugging Face Trainer.
Only LoRA adapter weights are updated.

In [30]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=data_collator,
)

trainer.train()

KeyError: 'audio'

## Save Fine-Tuned Adapters

We save the LoRA adapters (small files) for later inference and evaluation.
The base model remains unchanged.

In [ ]:
model.save_pretrained("./voxtral-esd-peft-adapters")
print("Saved adapters to ./voxtral-esd-peft-adapters")

## Quick Sanity Check Inference

We run inference on a single test sample to ensure the model produces
a label-like output after fine-tuning.

In [ ]:
sample = test_records[0]
audio, sr = sf.read(sample["audio"])

inputs = processor(
    audio=audio,
    sampling_rate=sr,
    text=sample["prompt"],
    return_tensors="pt"
).to(device)

model.eval()
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=5)

decoded = processor.tokenizer.decode(out[0], skip_special_tokens=True)
print("Prompt:", sample["prompt"])
print("GT label:", sample["label"])
print("Model output:", decoded)
